# Chain of Thought - 第二部分：Few-shot CoT

## 学习目标
1. 掌握 Few-shot CoT 的原理和使用
2. 学会设计高质量的 CoT 示例
3. 理解示例选择的最佳实践

## 目录
1. [Few-shot CoT 原理](#1-few-shot-cot-原理)
2. [创建高质量示例](#2-创建高质量示例)
3. [FewShotCoT 类使用](#3-fewshotcot-类使用)
4. [示例选择策略](#4-示例选择策略)
5. [实战案例](#5-实战案例)
6. [练习](#6-练习)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.chain_of_thought import (
    CoTExample, CoTResult, CoTPromptBuilder, FewShotCoT
)
print("模块加载成功！")

---
## 1. Few-shot CoT 原理

### 1.1 核心思想

Few-shot CoT 通过提供带有推理链的示例，引导模型学习推理模式。

In [ ]:
print("""
Few-shot CoT 工作原理：

┌─────────────────────────────────────────────────────┐
│  示例1: 问题 → 推理步骤 → 答案                        │
│  示例2: 问题 → 推理步骤 → 答案                        │
│  示例3: 问题 → 推理步骤 → 答案                        │
│  ─────────────────────────────────                  │
│  新问题: ???                                         │
│  模型学习模式后: 问题 → 推理步骤 → 答案               │
└─────────────────────────────────────────────────────┘

关键优势：
  1. 示例提供推理模板
  2. 模型学习推理格式和风格
  3. 效果比 Zero-shot 更稳定
""")

### 1.2 与 Zero-shot 对比

In [ ]:
print("""
┌──────────────┬─────────────────┬─────────────────┐
│     特性     │   Zero-shot     │    Few-shot     │
├──────────────┼─────────────────┼─────────────────┤
│ 示例数量     │ 0               │ 2-8个           │
│ 准备工作     │ 无              │ 需设计示例      │
│ 推理质量     │ 不稳定          │ 稳定可控        │
│ 格式一致性   │ 低              │ 高              │
│ 领域适应     │ 差              │ 好              │
│ Token消耗    │ 少              │ 多              │
└──────────────┴─────────────────┴─────────────────┘
""")

---
## 2. 创建高质量示例

### 2.1 示例设计原则

In [ ]:
print("""
高质量 CoT 示例的特征：

1. 清晰的步骤分解
   ✓ 每步只做一件事
   ✓ 步骤之间有逻辑连接
   ✗ 跳跃式推理

2. 明确的中间结果
   ✓ 显示计算过程和结果
   ✓ 标注关键数值
   ✗ 只给最终答案

3. 一致的格式
   ✓ 统一的步骤编号
   ✓ 相似的表达方式
   ✗ 格式混乱

4. 适当的复杂度
   ✓ 与目标问题难度相近
   ✗ 过于简单或复杂
""")

### 2.2 创建示例

In [ ]:
# 示例1：基础算术
example1 = CoTExample(
    question="小明有15元，买了一本8元的书，还剩多少钱？",
    reasoning="""让我一步步计算：
1. 小明原有：15元
2. 买书花费：8元
3. 剩余金额：15 - 8 = 7元""",
    answer="7元"
)

print("示例1 - 基础算术：")
print(example1.format())

In [ ]:
# 示例2：多步计算
example2 = CoTExample(
    question="商店有50个苹果，卖出20个，又进货30个，现在有多少？",
    reasoning="""让我一步步计算：
1. 初始数量：50个
2. 卖出后：50 - 20 = 30个
3. 进货后：30 + 30 = 60个""",
    answer="60个"
)

print("示例2 - 多步计算：")
print(example2.format())

In [ ]:
# 示例3：百分比计算
example3 = CoTExample(
    question="一件衣服原价200元，打8折后多少钱？",
    reasoning="""让我一步步计算：
1. 原价：200元
2. 8折 = 80% = 0.8
3. 折后价：200 × 0.8 = 160元""",
    answer="160元"
)

print("示例3 - 百分比：")
print(example3.format())

### 2.3 好示例 vs 坏示例

In [ ]:
# 坏示例：推理不清晰
bad_example = CoTExample(
    question="小明有15元，买了一本8元的书，还剩多少钱？",
    reasoning="15减8等于7",  # 太简单，没有步骤
    answer="7元"
)

print("坏示例（推理太简单）：")
print(bad_example.format())
print("\n问题：没有展示推理过程，模型无法学习")

In [ ]:
# 好示例：清晰的步骤
good_example = CoTExample(
    question="小明有15元，买了一本8元的书，还剩多少钱？",
    reasoning="""让我一步步计算：
1. 确定初始金额：小明有15元
2. 确定支出：买书花费8元
3. 计算剩余：15 - 8 = 7元""",
    answer="7元"
)

print("好示例（清晰步骤）：")
print(good_example.format())
print("\n优点：步骤清晰，格式一致，易于模型学习")

---
## 3. FewShotCoT 类使用

### 3.1 基本使用

In [ ]:
# 创建 FewShotCoT 实例
few_shot = FewShotCoT(examples=[example1, example2, example3])

print(f"FewShotCoT 配置：")
print(f"  示例数量: {len(few_shot._examples)}")

In [ ]:
# 生成提示
new_question = "小红有100元，买了3本书每本25元，还剩多少钱？"
prompt = few_shot.get_prompt(new_question)

print("Few-shot CoT 完整提示：")
print("="*60)
print(prompt)
print("="*60)

### 3.2 添加和管理示例

In [ ]:
# 添加新示例
new_example = CoTExample(
    question="一个班有30人，男生比女生多6人，男生有多少人？",
    reasoning="""让我一步步计算：
1. 总人数：30人
2. 设女生x人，则男生(x+6)人
3. 方程：x + (x+6) = 30
4. 解方程：2x = 24，x = 12
5. 男生：12 + 6 = 18人""",
    answer="18人"
)

few_shot.add_example(new_example)
print(f"添加后示例数量: {len(few_shot._examples)}")

---
## 4. 示例选择策略

### 4.1 多样性原则

In [ ]:
print("""
示例选择策略：

1. 多样性
   - 覆盖不同问题类型
   - 包含不同难度级别
   - 展示不同推理模式

2. 相关性
   - 与目标问题领域相关
   - 使用相似的概念和术语

3. 数量
   - 通常 3-8 个示例最佳
   - 太少：模式不明显
   - 太多：Token浪费，可能混淆

4. 顺序
   - 简单到复杂
   - 或按相关性排序
""")

In [ ]:
# 创建多样化示例集
diverse_examples = [
    # 简单加减
    CoTExample(
        question="5 + 3 = ?",
        reasoning="直接计算：5 + 3 = 8",
        answer="8"
    ),
    # 多步计算
    CoTExample(
        question="(5 + 3) × 2 = ?",
        reasoning="""1. 先算括号：5 + 3 = 8
2. 再乘2：8 × 2 = 16""",
        answer="16"
    ),
    # 应用题
    CoTExample(
        question="苹果3元/个，买5个多少钱？",
        reasoning="""1. 单价：3元
2. 数量：5个
3. 总价：3 × 5 = 15元""",
        answer="15元"
    ),
]

print("多样化示例集：")
for i, ex in enumerate(diverse_examples, 1):
    print(f"\n示例{i}: {ex.question}")

---
## 5. 实战案例

### 5.1 数学应用题

In [ ]:
# 数学应用题示例集
math_examples = [
    CoTExample(
        question="火车2小时行驶160公里，平均速度是多少？",
        reasoning="""1. 距离：160公里
2. 时间：2小时
3. 速度 = 距离/时间 = 160/2 = 80公里/小时""",
        answer="80公里/小时"
    ),
    CoTExample(
        question="长方形长8cm宽5cm，面积是多少？",
        reasoning="""1. 长：8cm
2. 宽：5cm
3. 面积 = 长×宽 = 8×5 = 40平方厘米""",
        answer="40平方厘米"
    ),
]

math_cot = FewShotCoT(examples=math_examples)
prompt = math_cot.get_prompt("圆的半径是7cm，周长是多少？(π取3.14)")
print("数学应用题提示：")
print(prompt)

### 5.2 逻辑推理

In [ ]:
# 逻辑推理示例集
logic_examples = [
    CoTExample(
        question="所有鸟都会飞。企鹅是鸟。企鹅会飞吗？",
        reasoning="""1. 前提1：所有鸟都会飞
2. 前提2：企鹅是鸟
3. 根据三段论：企鹅会飞
4. 但实际上前提1是错误的，企鹅不会飞""",
        answer="根据给定前提会飞，但实际不会"
    ),
    CoTExample(
        question="如果下雨，地面会湿。地面湿了，一定下雨了吗？",
        reasoning="""1. 条件：下雨 → 地面湿
2. 观察：地面湿
3. 这是"肯定后件"谬误
4. 地面湿可能有其他原因（洒水、露水等）""",
        answer="不一定，可能有其他原因"
    ),
]

logic_cot = FewShotCoT(examples=logic_examples)
print("逻辑推理示例已创建")

---
## 6. 练习

### 练习1：创建示例集

In [ ]:
# TODO: 为"时间计算"创建3个示例
# 例如：现在8点，3小时后几点？

# time_examples = [
#     CoTExample(...),
#     CoTExample(...),
#     CoTExample(...),
# ]

### 练习2：使用 FewShotCoT

In [ ]:
# TODO: 使用你创建的示例生成提示
# time_cot = FewShotCoT(examples=time_examples)
# prompt = time_cot.get_prompt("现在下午2点，5小时后几点？")
# print(prompt)

---
## 下一步

继续学习 **01c_ChainOfThought_Advanced.ipynb** 了解 Auto-CoT 和高级技巧